In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset,TensorDataset
from tensorflow.keras import layers, models
from tensorflow.keras import optimizers


In [ ]:
X = pd.read_csv("X_train_9xQjqvZ.csv", index_col="ROW_ID")
y = pd.read_csv("y_train_Ppwhaz8.csv", index_col="ROW_ID")
X_test = pd.read_csv("X_test_1zTtEnD.csv")


In [3]:
nan_par_colonne = X.isna().sum().sort_values(ascending=False)
print(nan_par_colonne[nan_par_colonne > 100]/len(X)*100)

SIGNED_VOLUME_1          73.520366
SIGNED_VOLUME_20          1.612300
MEDIAN_DAILY_TURNOVER     0.690227
SIGNED_VOLUME_19          0.569561
SIGNED_VOLUME_18          0.162976
dtype: float64


In [4]:
scaler = StandardScaler()
label_enc = LabelEncoder()
y_scaler = StandardScaler()
ret_cols = [f"RET_{i}" for i in range(1, 21)]
vol_cols = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
turnover_median = None

In [5]:
def augment_data(X, train = True):
    global median_turnover
    X_out = X.copy()

    # add NAN indicator for SIGNED_VOLUME_1
    X_out["IS_NAN_SIGNED_VOLUME_1"] = (
        X_out["SIGNED_VOLUME_1"].isna().astype(float)
    )

    # replace NAN by 0
    X_out[ret_cols] = X_out[ret_cols].fillna(0.0)
    X_out[vol_cols] = X_out[vol_cols].fillna(0.0)

    # replace MEDIAN NAN by train median 
    if train:
        median_turnover = X_out["MEDIAN_DAILY_TURNOVER"].median()
    X_out["MEDIAN_DAILY_TURNOVER"] = X_out["MEDIAN_DAILY_TURNOVER"].fillna(
        median_turnover
    )

    # add new features
    RET_features = [f'RET_{i}' for i in range(1,21)]
    SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,21)]

    for i in range(1, 21):
        X_out[ f'AVERAGE_PERF_{i}'] = X_out[RET_features[:i]].mean(axis=1)
        X_out[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_out.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
        X_out[f"RET_VS_MKT_{i}"] = X_out[f"RET_{i}"] - X_out[f"ALLOCATIONS_AVERAGE_PERF_{i}"]
        X_out[f"VOL_VS_MKT_{i}"] = X_out[f"SIGNED_VOLUME_{i}"] - X_out.groupby("TS")[f"SIGNED_VOLUME_{i}"].transform("mean")

    """ for i in range(1,21):
        X_out[ f'STD_PERF_{i}'] = X_out[RET_features[:i]].std(1)
        X_out[ f'ALLOCATIONS_STD_PERF_{i}'] = X_out.groupby('TS')[ f'STD_PERF_{i}'].transform('mean')"""



    return X_out

In [6]:
augmented_X = augment_data(X, train=True)
# Transforme les variables en quantiles uniformes [-0.5, 0.5] chaque jour
for col in augmented_X.columns:
    augmented_X[col] = augmented_X.groupby('TS')[col].rank(pct=True) - 0.5

C:\Users\antoi\AppData\Local\Temp\ipykernel_22276\1589891598.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_out[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_out.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
C:\Users\antoi\AppData\Local\Temp\ipykernel_22276\1589891598.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_out[f"RET_VS_MKT_{i}"] = X_out[f"RET_{i}"] - X_out[f"ALLOCATIONS_AVERAGE_PERF_{i}"]
C:\Users\antoi\AppData\Local\Temp\ipykernel_22276\1589891598.py:29: PerformanceWarning: DataFrame is highly fragmented

In [7]:
augmented_X

,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,RET_13,...,RET_VS_MKT_18,VOL_VS_MKT_18,AVERAGE_PERF_19,ALLOCATIONS_AVERAGE_PERF_19,RET_VS_MKT_19,VOL_VS_MKT_19,AVERAGE_PERF_20,ALLOCATIONS_AVERAGE_PERF_20,RET_VS_MKT_20,VOL_VS_MKT_20
ROW_ID,,,,,,,,,,,,,,,,,,,,,
0,0.001812,-0.498187,-0.499408,-0.063464,-0.483128,-0.349710,0.091610,-0.226584,-0.362179,0.381001,...,-0.480734,0.219987,-0.442937,-0.462685,-0.004893,0.230556,-0.490795,-0.491750,-0.499345,0.366745
1,0.001812,-0.494564,-0.478309,-0.193253,0.255536,0.001617,0.380227,0.278396,0.155054,-0.330797,...,0.293416,-0.478483,0.377583,-0.462685,-0.142278,-0.477396,0.232268,-0.491750,-0.474194,-0.437771
2,0.001812,-0.490941,-0.499189,-0.449298,-0.465547,0.446065,0.481585,-0.474029,-0.461215,-0.256131,...,-0.460196,-0.066746,-0.468072,-0.462685,-0.441403,0.204663,-0.493590,-0.491750,-0.499106,0.414253
3,0.001812,-0.487317,-0.465297,0.374910,0.484853,-0.442991,-0.461984,-0.068624,-0.028499,-0.464446,...,0.487152,-0.344892,-0.442467,-0.462685,0.394669,-0.330316,-0.465601,-0.491750,-0.458488,-0.350487
4,0.001812,-0.483694,-0.495788,-0.468089,0.489481,-0.497868,0.329155,0.406594,0.088220,-0.417156,...,0.490992,-0.154152,0.380418,-0.462685,-0.463353,-0.127782,0.100342,-0.491750,-0.495130,0.117960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527068,0.001812,0.483697,-0.108249,0.278783,0.310393,0.466947,0.412081,0.290419,-0.423458,0.241649,...,0.333510,-0.065602,0.380506,-0.392772,0.302424,0.112316,0.366531,-0.392772,-0.071012,0.121828
527069,0.001812,0.487320,0.416060,-0.451419,-0.285389,-0.015019,0.424693,0.357162,0.480327,0.267066,...,-0.255769,0.001443,0.178043,-0.392772,-0.446284,0.138805,0.276245,-0.392772,0.424969,0.197770
527070,0.001812,0.490943,0.465672,0.102602,-0.269565,-0.388999,0.081184,0.457544,0.454976,0.302424,...,-0.238405,0.284786,-0.079852,-0.392772,0.139272,0.150771,0.124503,-0.392772,0.469344,-0.032094


In [8]:
features_by_importance = ['RET_VS_MKT_1', 'RET_VS_MKT_4','ALLOCATIONS_AVERAGE_PERF_16',
       'ALLOCATIONS_AVERAGE_PERF_3', 'ALLOCATIONS_AVERAGE_PERF_1',
       'ALLOCATIONS_AVERAGE_PERF_14', 'ALLOCATIONS_AVERAGE_PERF_20', 'RET_VS_MKT_10',
       'ALLOCATIONS_AVERAGE_PERF_7', 'ALLOCATIONS_AVERAGE_PERF_5',
       'ALLOCATIONS_AVERAGE_PERF_19', 'ALLOCATIONS_AVERAGE_PERF_12', 'RET_17',
       'ALLOCATIONS_AVERAGE_PERF_4', 'AVERAGE_PERF_4',
       'ALLOCATIONS_AVERAGE_PERF_6', 'ALLOCATIONS_STD_PERF_3', 'RET_7',
       'SIGNED_VOLUME_1', 'ALLOCATIONS_STD_PERF_20']

In [9]:


num_cols = (
    [f"RET_{i}" for i in range(1, 21)]
    + [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
    + [f"ALLOCATIONS_AVERAGE_PERF_{i}" for i in range(1, 21)]
    + ["MEDIAN_DAILY_TURNOVER","IS_NAN_SIGNED_VOLUME_1"]
)
num_cols = (features_by_importance[:10] + ["VOL_VS_MKT_1","VOL_VS_MKT_4"])


def prepare_inputs(data, train=True):
    if train:
        X_num = scaler.fit_transform(data[num_cols].values)
        X_cat = label_enc.fit_transform(data["GROUP"].values)
    else:
        X_num = scaler.transform(data[num_cols].values)
        X_cat = label_enc.transform(data["GROUP"].values)

    return (
        torch.tensor(X_num, dtype=torch.float32),
        torch.tensor(X_cat, dtype=torch.long),
    )

In [10]:
X_num, X_cat = prepare_inputs(augmented_X, train=True)
by = torch.tensor(y["target"].values, dtype=torch.float32)
by_scaled = torch.tensor(y_scaler.fit_transform(y.values.reshape(-1, 1)), dtype=torch.float32).squeeze()

market_target_mean = y.groupby(X['TS'])['target'].transform('mean')
y_train_rel = y['target'] - market_target_mean
by_scaled = torch.tensor(y_scaler.fit_transform(y_train_rel.values.reshape(-1, 1)), dtype=torch.float32).squeeze()


# Split Train / Validation
train_idx, val_idx = train_test_split(
    np.arange(len(augmented_X)), test_size=0.2, random_state=42, shuffle=True
)

train_loader = DataLoader(
    TensorDataset(X_num[train_idx], X_cat[train_idx], by_scaled[train_idx]),
    batch_size=128,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_num[val_idx], X_cat[val_idx], by_scaled[val_idx]),
    batch_size=1024,
    shuffle=False,
)
print(by)
by_scaled

tensor([0.0092, 0.0004, 0.0060,  ..., 0.0040, 0.0016, 0.0006])


tensor([ 2.8250, -0.1084,  1.7445,  ...,  1.3874,  0.5882,  0.2347])

In [11]:
num_cols

['RET_VS_MKT_1',
 'RET_VS_MKT_4',
 'ALLOCATIONS_AVERAGE_PERF_16',
 'ALLOCATIONS_AVERAGE_PERF_3',
 'ALLOCATIONS_AVERAGE_PERF_1',
 'ALLOCATIONS_AVERAGE_PERF_14',
 'ALLOCATIONS_AVERAGE_PERF_20',
 'RET_VS_MKT_10',
 'ALLOCATIONS_AVERAGE_PERF_7',
 'ALLOCATIONS_AVERAGE_PERF_5',
 'VOL_VS_MKT_1',
 'VOL_VS_MKT_4']

In [45]:
class TabularMLP(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        in_dim = num_features

        # Bloc 1
        self.block1 = nn.Sequential(
            nn.Linear(in_dim, 20),
            nn.BatchNorm1d(20),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        # Bloc 2
        self.block2 = nn.Sequential(
            nn.Linear(20, 20),
            nn.BatchNorm1d(20),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        self.block3 = nn.Sequential(
                    nn.Linear(20, 32),
                    nn.BatchNorm1d(32),
                    nn.ReLU(),
                    nn.Dropout(0.5),
                )

        self.head = nn.Linear(32, 1)

    def forward(self, x_num, x_cat):
        if self.training:
            x_num = x_num + torch.randn_like(x_num) * 0.01


        x = self.block1(x_num)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x).squeeze(-1)


In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TabularMLP(
    num_features=X_num.shape[1]
).to(device)

class PearsonCorrelationLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, target):
        x = preds - preds.mean()
        y = target - target.mean()
        r = (x * y).sum() / (torch.sqrt((x ** 2).sum() + 1e-8) * torch.sqrt((y ** 2).sum() + 1e-8))
        return -r
criterion = nn.HuberLoss(delta=1.0)
#criterion = PearsonCorrelationLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

In [50]:
epochs = 15
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    train_correct, train_total = 0, 0

    for x_n, x_c, target in train_loader:
        x_n, x_c, target = x_n.to(device), x_c.to(device), target.to(device)

        optimizer.zero_grad()
        preds = model(x_n, x_c)
        loss = criterion(preds, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(target)

        pred_sign = (preds.detach() > 0).int()
        true_sign = (target > 0).int()
        train_correct += (pred_sign == true_sign).sum().item()
        train_total += target.size(0)
    train_acc = train_correct / train_total

    # validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_n, x_c, target in val_loader:
            x_n, x_c, target = x_n.to(device), x_c.to(device), target.to(device)
            preds = model(x_n, x_c)

            pred_sign = (preds > 0).int()
            true_sign = (target > 0).int()

            correct += (pred_sign == true_sign).sum().item()
            total += target.size(0)

    val_acc = correct / total
    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {total_loss / len(train_idx):.6f} | "
        f"Train Accuracy: {train_acc:.4f} | "
        f"Val Accuracy: {val_acc:.4f}"
    )

Epoch 1/15 | Train Loss: 0.370959 | Train Accuracy: 0.5191 | Val Accuracy: 0.5222
Epoch 2/15 | Train Loss: 0.370942 | Train Accuracy: 0.5185 | Val Accuracy: 0.5225
Epoch 3/15 | Train Loss: 0.371000 | Train Accuracy: 0.5190 | Val Accuracy: 0.5223
Epoch 4/15 | Train Loss: 0.370988 | Train Accuracy: 0.5188 | Val Accuracy: 0.5224
Epoch 5/15 | Train Loss: 0.371026 | Train Accuracy: 0.5187 | Val Accuracy: 0.5220
Epoch 6/15 | Train Loss: 0.370987 | Train Accuracy: 0.5184 | Val Accuracy: 0.5217
Epoch 7/15 | Train Loss: 0.370998 | Train Accuracy: 0.5187 | Val Accuracy: 0.5217
Epoch 8/15 | Train Loss: 0.370983 | Train Accuracy: 0.5191 | Val Accuracy: 0.5222
Epoch 9/15 | Train Loss: 0.370959 | Train Accuracy: 0.5196 | Val Accuracy: 0.5220
Epoch 10/15 | Train Loss: 0.370984 | Train Accuracy: 0.5189 | Val Accuracy: 0.5220
Epoch 11/15 | Train Loss: 0.370992 | Train Accuracy: 0.5185 | Val Accuracy: 0.5215
Epoch 12/15 | Train Loss: 0.370961 | Train Accuracy: 0.5176 | Val Accuracy: 0.5211
Epoch 13/15 |

In [43]:
pred = model(X_num, X_cat)
pred = (preds > 0).int().numpy()

print(np.unique_counts(pred)[1][1]/np.unique_counts(pred)[1].sum())
print(np.unique_counts(y["target"] > 0)[1][1]/np.unique_counts(y["target"] > 0)[1].sum())

0.4984488107549121
0.5071840143585423


In [ ]:
augmented_X_test = augment_data(X_test, train=False)
for col in augmented_X_test.columns:
    augmented_X_test[col] = augmented_X_test.groupby('TS')[col].rank(pct=True) - 0.5
X_test_num, X_test_cat = prepare_inputs(augmented_X_test, train=False)

model.eval()
preds_proba = model(
    X_test_num, X_test_cat
)
preds = preds_proba.detach().cpu().numpy()
preds = (preds > 0).astype(int)
preds_binary = (preds > 0).astype(int)

sub = pd.DataFrame({"ROW_ID": X_test["ROW_ID"], "target": preds_binary})
sub.to_csv("y.csv", index=False)

C:\Users\antoi\AppData\Local\Temp\ipykernel_12592\1589891598.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_out[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_out.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
C:\Users\antoi\AppData\Local\Temp\ipykernel_12592\1589891598.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_out[f"RET_VS_MKT_{i}"] = X_out[f"RET_{i}"] - X_out[f"ALLOCATIONS_AVERAGE_PERF_{i}"]
C:\Users\antoi\AppData\Local\Temp\ipykernel_12592\1589891598.py:29: PerformanceWarning: DataFrame is highly fragmented

ValueError: y contains previously unseen labels: [np.float64(-0.41383495145631066), np.float64(-0.3876201923076923), np.float64(-0.37408073541167064), np.float64(-0.37303743961352653), np.float64(-0.3321428571428572), np.float64(-0.24642857142857144), np.float64(-0.24393203883495146), np.float64(-0.14923878205128205), np.float64(-0.12228217426059151), np.float64(-0.11941425120772947), np.float64(0.0011904761904761862), np.float64(0.0071428571428571175), np.float64(0.01092233009708743), np.float64(0.07512019230769229), np.float64(0.1295163868904876), np.float64(0.1342089371980676), np.float64(0.25357142857142856), np.float64(0.33452380952380956), np.float64(0.33673878205128205), np.float64(0.34101941747572817), np.float64(0.3777178257394085), np.float64(0.3805857487922706)]